# World Events MCP Server — Colab Test Notebook

This notebook installs and tests the **World Events MCP Server** — a real-time global intelligence platform covering 70+ tools across 30+ domains.

All data sources are **free public APIs** — no API keys required.

---
**Domains covered:** Markets · Seismology · Military · Infrastructure · Climate ·
Humanitarian · News · Cyber · Geopolitical Analysis · Strategic Synthesis

## 1. Install

In [1]:
# Clone the repo and install the package
!git clone https://github.com/dshipley71/WorldEventsPipeline.git 2>/dev/null || echo 'Already cloned'
!pip install -q -e '/content/WorldEventsPipeline/world-events-mcp'

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.4 MB/s eta 0:00:00
  Building editable for world-events-mcp (pyproject.toml) ... done


## 2. MCP Client Helper

The server speaks **JSON-RPC 2.0 over stdio**. This helper manages the subprocess and handles the protocol handshake.

In [2]:
import json
import subprocess
import threading
import queue
import time
import sys
from IPython.display import display, JSON


class MCPClient:
    """Minimal synchronous MCP client over stdio."""

    def __init__(self):
        self._proc = None
        self._reader_thread = None
        self._responses: queue.Queue = queue.Queue()
        self._stderr_lines: list[str] = []
        self._id = 0

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def connect(self):
        """Start the MCP server subprocess and run the initialize handshake."""
        self._proc = subprocess.Popen(
            ["world-events-mcp"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        # Background thread: read stdout lines and push to response queue
        self._reader_thread = threading.Thread(
            target=self._read_loop, daemon=True
        )
        self._reader_thread.start()

        # MCP initialize handshake
        resp = self._send("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "colab-tester", "version": "1.0"},
        })
        # Acknowledge initialization
        self._notify("notifications/initialized", {})
        print(f"✅ Connected — server: {resp.get('result', {}).get('serverInfo', {})}")
        return self

    def disconnect(self):
        """Terminate the server process."""
        if self._proc:
            self._proc.terminate()
            self._proc = None
        print("Disconnected.")

    def __enter__(self):
        return self.connect()

    def __exit__(self, *_):
        self.disconnect()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def list_tools(self) -> list[dict]:
        """Return all available tool definitions."""
        resp = self._send("tools/list", {})
        return resp.get("result", {}).get("tools", [])

    def call_tool(self, name: str, arguments: dict | None = None, timeout: float = 30.0) -> dict:
        """Invoke a named tool with the given arguments."""
        resp = self._send(
            "tools/call",
            {"name": name, "arguments": arguments or {}},
            timeout=timeout,
        )
        result = resp.get("result", {})
        # Unwrap MCP content envelope
        contents = result.get("content", [])
        for block in contents:
            if block.get("type") == "text":
                try:
                    return json.loads(block["text"])
                except json.JSONDecodeError:
                    return {"raw": block["text"]}
        return result

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _next_id(self) -> int:
        self._id += 1
        return self._id

    def _write(self, msg: dict):
        line = json.dumps(msg) + "\n"
        self._proc.stdin.write(line)
        self._proc.stdin.flush()

    def _send(self, method: str, params: dict, timeout: float = 30.0) -> dict:
        msg_id = self._next_id()
        self._write({"jsonrpc": "2.0", "id": msg_id, "method": method, "params": params})
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                resp = self._responses.get(timeout=1.0)
                if resp.get("id") == msg_id:
                    return resp
                # put back if it's for another id
                self._responses.put(resp)
            except queue.Empty:
                pass
        raise TimeoutError(f"No response for method={method!r} within {timeout}s")

    def _notify(self, method: str, params: dict):
        """Send a notification (no id, no response expected)."""
        self._write({"jsonrpc": "2.0", "method": method, "params": params})

    def _read_loop(self):
        for line in self._proc.stdout:
            line = line.strip()
            if not line:
                continue
            try:
                msg = json.loads(line)
                if "id" in msg:
                    self._responses.put(msg)
                # drop server-sent notifications
            except json.JSONDecodeError:
                pass


# Instantiate and connect once for the whole notebook
client = MCPClient()
client.connect()

✅ Connected — server: {'name': 'world-events-mcp', 'version': '1.26.0'}


## 3. List Available Tools

In [3]:
tools = client.list_tools()
print(f"Total tools: {len(tools)}\n")
for t in tools:
    print(f"  {t['name']:<40} {t.get('description', '')[:70]}")

Total tools: 70

  intel_macro_signals                      Get 7 key macro signals: Fear & Greed, mempool fees, DXY, VIX, gold, 1
  intel_world_bank_indicators              Get World Bank development indicators (GDP, inflation, unemployment) f
  intel_earthquakes                        Get recent earthquakes from USGS. No API key needed.
  intel_humanitarian_summary               Get humanitarian crisis datasets from HDX. No API key needed. Optional
  intel_military_flights                   Track military aircraft via OpenSky Network (ICAO hex + callsign filte
  intel_theater_posture                    Get military aircraft presence across 5 theaters: European, Indo-Pacif
  intel_aircraft_details                   Get aircraft details from hexdb.io by ICAO24 hex code (free, no API ke
  intel_internet_outages                   Get internet outages from IODA (Georgia Tech Internet Intelligence) — 
  intel_cable_health                       Assess undersea cable corridor health from NGA

## 4. Markets — Macro Signals
Fear & Greed Index and Bitcoin mempool fees (both free, no key).

In [4]:
result = client.call_tool("intel_macro_signals")
display(JSON(result))

<IPython.core.display.JSON object>

## 5. Seismology — Recent Earthquakes (USGS)

In [5]:
result = client.call_tool("intel_earthquakes", {"min_magnitude": 5.0, "hours": 48})

quakes = result.get("earthquakes", [])
print(f"M5.0+ earthquakes in the past 48 h: {len(quakes)}\n")
for q in quakes[:10]:
    print(f"  M{q.get('magnitude','?'):>4}  {q.get('place',''):<50}  {q.get('time','')[:19]}")

M5.0+ earthquakes in the past 48 h: 15

  M 5.5  2 km SSE of Rodotópi, Greece                        2026-03-08T03:32:31
  M 5.1  20 km ESE of Farkhār, Afghanistan                   2026-03-07T18:24:35
  M 5.1  119 km E of Caucete, Argentina                      2026-03-07T17:13:26
  M 5.2  95 km SSE of Sungai Penuh, Indonesia                2026-03-07T16:58:43
  M   5  4 km E of Hukay, Philippines                        2026-03-07T16:04:54
  M 5.1  93 km E of Hihifo, Tonga                            2026-03-07T00:44:53
  M 5.3  71 km SSE of Shihezi, China                         2026-03-06T21:44:46
  M 5.7  228 km ESE of Attu Station, Alaska                  2026-03-06T21:08:19
  M 5.3  66 km E of Tobelo, Indonesia                        2026-03-06T17:55:53
  M 5.1  132 km ENE of Masohi, Indonesia                     2026-03-06T15:27:47


## 6. Military — Hotspot Escalation Scoring

In [6]:
result = client.call_tool("intel_hotspot_escalation")

hotspots = result.get("hotspots", [])
print(f"Hotspots scored (0–100):\n")
for h in sorted(hotspots, key=lambda x: x.get("score", 0), reverse=True)[:15]:
    bar = "█" * int(h.get("score", 0) / 5)
    print(f"  {h.get('name',''):<30} {h.get('score',0):>3}  {bar}")

Hotspots scored (0–100):

  kyiv                           20.0  ████
  gaza                           20.0  ████
  khartoum                       20.0  ████
  damascus                       16.0  ███
  sanaa                          16.0  ███
  mogadishu                      16.0  ███
  beirut                         16.0  ███
  crimea                         16.0  ███
  bab_el_mandeb                  16.0  ███
  tehran                         12.0  ██
  taipei                         12.0  ██
  pyongyang                      12.0  ██
  kabul                          12.0  ██
  naypyidaw                      12.0  ██
  baghdad                        12.0  ██


## 7. News — Feed with Category Filter

In [8]:
result = client.call_tool("intel_news_feed", {"categories": ["geopolitics", "military"], "limit": 15})

articles = result.get("articles", [])
print(f"Latest {len(articles)} articles:\n")
for a in articles:
    print(f"  [{a.get('category',''):<12}] {a.get('title','')[:80]}")
    print(f"    {a.get('published','')[:19]}  {a.get('source','')}")
    print()

Latest 0 articles:



## 8. Cyber — Threat Intelligence Feeds

In [9]:
result = client.call_tool("intel_cyber_threats")

# CISA Known Exploited Vulnerabilities
kev = result.get("cisa_kev", {})
print(f"CISA KEV catalog — total: {kev.get('total', '?')}, recent (30d): {kev.get('recent_30d', '?')}")
print()

for vuln in kev.get("recent_vulnerabilities", [])[:5]:
    print(f"  {vuln.get('cveID',''):<20} {vuln.get('vendorProject',''):<20} {vuln.get('vulnerabilityName','')[:45]}")

CISA KEV catalog — total: ?, recent (30d): ?



## 9. Climate — Space Weather (NOAA)

In [10]:
result = client.call_tool("intel_space_weather")
display(JSON(result))

<IPython.core.display.JSON object>

## 10. Strategic — Global Risk Posture (9-Domain Weighted Score)

In [11]:
result = client.call_tool("intel_strategic_posture", timeout=60.0)

overall = result.get("overall_risk", {})
print(f"Overall risk score: {overall.get('score', '?')} / 100  ({overall.get('level', '?')})\n")

domains = result.get("domains", {})
print("Domain scores:")
for domain, data in sorted(domains.items(), key=lambda x: x[1].get("score", 0), reverse=True):
    score = data.get("score", 0)
    bar = "█" * int(score / 5)
    print(f"  {domain:<30} {score:>3}  {bar}")

Overall risk score: ? / 100  (?)

Domain scores:


## 11. Country Intelligence — Instability Index

In [13]:
# Try a few countries of interest
for iso in ["SDN", "SYR", "IRN", "UKR", "DEU"]:
    result = client.call_tool("intel_instability_index", {"country": iso})
    score = result.get("score", "?")
    level = result.get("level", "?")
    name  = result.get("country_name", iso)
    bar = "█" * (int(score) // 5) if isinstance(score, (int, float)) else ""
    print(f"  {name:<25} {iso}  score={score:>3}  [{level:<8}]  {bar}")

TimeoutError: No response for method='tools/call' within 30.0s

## 12. Cross-Domain Alert Digest
Threshold-based alerts across 7 intelligence domains.

In [14]:
result = client.call_tool("intel_alert_digest", timeout=60.0)

alerts = result.get("alerts", [])
print(f"Active alerts: {len(alerts)}\n")
for alert in alerts[:20]:
    sev = alert.get("severity", "?")
    domain = alert.get("domain", "?")
    msg = alert.get("message", "")
    print(f"  [{sev:<8}] {domain:<20} {msg[:70]}")

Active alerts: 3

  [?       ] political            3 countries above instability threshold (>=70)
  [?       ] military             1 military surge anomalies detected
  [?       ] infrastructure       5 cable corridors at elevated risk: transatlantic_north, transatlantic


## 13. Server Health — Data Source Status

In [15]:
result = client.call_tool("intel_status")

sources = result.get("sources", {})
cache   = result.get("cache", {})
cb      = result.get("circuit_breakers", {})

print(f"Cache entries: {cache.get('entries', '?')}  |  "
      f"Cache size: {cache.get('size_mb', '?')} MB\n")

open_breakers = [k for k, v in cb.items() if v.get("state") == "open"]
if open_breakers:
    print(f"⚠️  Open circuit breakers: {', '.join(open_breakers)}\n")
else:
    print("✅ All circuit breakers closed\n")

for src, info in list(sources.items())[:20]:
    status = "✅" if info.get("healthy") else "❌"
    print(f"  {status} {src:<35} last_used={info.get('last_used', 'never')}")

Cache entries: ?  |  Cache size: ? MB

✅ All circuit breakers closed



AttributeError: 'list' object has no attribute 'get'

## 14. Custom Tool Call
Call any tool by name with arbitrary arguments.

In [16]:
# ---- Customize these ----
TOOL_NAME = "intel_gdelt_search"
ARGUMENTS = {"query": "taiwan strait", "mode": "ArtList", "max_records": 10}
# -------------------------

result = client.call_tool(TOOL_NAME, ARGUMENTS, timeout=30.0)
display(JSON(result))

<IPython.core.display.JSON object>

## 15. Disconnect

In [17]:
client.disconnect()

Disconnected.
